In [1]:
# 1. Imports

from pathlib import Path
import time


# ============================================================
# EXTERNE BIBLIOTHEKEN - Datenverarbeitung & Analyse
# ============================================================
import numpy as np                           # Numerische Berechnungen
import pandas as pd                          # Datenanalyse und TimeSeriesManagement
import ccxt                                  # Kryptobörsen-API (Binance etc.)

from dataclasses import dataclass
import configparser
import hmac
import hashlib
import requests
import pyarrow

In [2]:
# 2. Configuration

# Change run date
RUN_DATE = "2026-03-13"
TOP_N = 100
VS_CURRENCY = "usd"


PROJECT_ROOT = Path.cwd().parent.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
FETCHED_DIR = RAW_DIR / "fetched"
PROCESSED_DIR = DATA_DIR / "processed"

ASSETS_PATH = FETCHED_DIR / "assets.parquet"
ASSET_DAILY_PATH = FETCHED_DIR / "asset_daily"  
ASSET_OHLCV_5M_DIR = FETCHED_DIR / "asset_ohlcv_5m"

SRC_PATH = PROJECT_ROOT / "Src"
SECRET_PATH = SRC_PATH / "secret.cfg"

TARGET_VOLUME_EXCHANGE = "binance"
USD_LIKE_QUOTES = ["USDT", "USDC", "USD"]

OHLCV_INTERVAL = "5m"
BINANCE_KLINE_LIMIT = 1000
OHLCV_START_DATE = RUN_DATE
OHLCV_END_DATE_EXCLUSIVE = "2026-03-14"

def controlpath():
    print(f"Projektverzeichnis:            {PROJECT_ROOT}")
    print(f"Datenverzeichnis:              {DATA_DIR}")
    print(f"Rohdatenverzeichnis:           {RAW_DIR}")
    print(f"Gefetchte Daten:               {FETCHED_DIR}")
    print(f"Verarbeitete Daten:            {PROCESSED_DIR}")
    print(f"Source-Code:                   {SRC_PATH}")
    print(f"Secret:                        {SECRET_PATH}")
    print()

    checks = {
        "Assets-Datei":              ASSETS_PATH,
        "Asset-Daily-Verzeichnis":   ASSET_DAILY_PATH,
        "Asset-OHLCV-5M-Verzeichnis": ASSET_OHLCV_5M_DIR,
    }

    for name, path in checks.items():
        exists = path.exists()
        symbol = "✅" if exists else "❌ FEHLT"
        print(f"  {symbol}  {name}: {path}")

        # Falls Ordner → Inhalt anzeigen
        if exists and path.is_dir():
            files = sorted(path.glob("*.parquet"))
            if files:
                for f in files:
                    print(f"       📄 {f.name}")
            else:
                print(f"       ⚠️  Ordner ist leer — keine .parquet Dateien!")


OHLCV_5M_COLUMNS = [
    "timestamp",
    "asset_id",
    "exchange",
    "market_symbol",
    "open",
    "high",
    "low",
    "close",
    "volume_base",
    "volume_quote",
    "trade_count",
]

controlpath()

Projektverzeichnis:            c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit
Datenverzeichnis:              c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data
Rohdatenverzeichnis:           c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw
Gefetchte Daten:               c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched
Verarbeitete Daten:            c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\processed
Source-Code:                   c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\Src
Secret:                        c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\Src\secret.cfg

  ✅  Assets-Datei: c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched\assets.parquet
  ✅  Asset-Daily-Verzeichnis: c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched\asset_daily
       📄 asset_daily.parquet
  ✅  Asset-

In [26]:
# =========================
# Konfiguration
# =========================
BASE_URL               = "https://data-api.binance.vision"
INTERVAL               = "5m"
SLEEP_BETWEEN_REQUESTS = 0.15

# =========================
# Public REST
# =========================
SESSION = requests.Session()

def public_get(path: str, params: dict | None = None):
    """
    Öffentlicher GET-Request an Binance API.
    Kein API-Key nötig — /api/v3/klines ist ein Public Endpoint.
    """
    url = f"{BASE_URL}{path}"
    r = SESSION.get(url, params=params, timeout=20)
    if r.status_code >= 400:
        raise RuntimeError(f"Binance Fehler {r.status_code}: {r.text}")
    return r.json()


# =========================
# Parquet Speicherung
# =========================
def json_to_parquet(data: list[dict]) -> None:
    """
    Speichert OHLCV-Daten als Parquet — pro Monat eine Datei.
    - Datei existiert bereits → zusammenführen + deduplizieren
    - Datei existiert nicht   → neu erstellen
    Sortierung: market_symbol → timestamp
    """
    ASSET_OHLCV_5M_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.DataFrame(data)
    df["timestamp"] = df["timestamp"].dt.tz_convert("UTC")
    df["_year"]     = df["timestamp"].dt.year
    df["_month"]    = df["timestamp"].dt.month

    for (year, month), month_df in df.groupby(["_year", "_month"]):
        month_df = month_df.drop(columns=["_year", "_month"])
        out      = ASSET_OHLCV_5M_DIR / f"ohlcv_{year:04d}-{month:02d}.parquet"

        if out.exists():
            df_all = (
                pd.concat([pd.read_parquet(out), month_df], ignore_index=True)
                .drop_duplicates(subset=["timestamp", "market_symbol"])
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            print(f"  [PARQUET] Aktualisiert: {out.name} ({len(df_all)} Zeilen)")
        else:
            df_all = (
                month_df
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            print(f"  [PARQUET] Erstellt: {out.name} ({len(df_all)} Zeilen)")

        df_all.to_parquet(out, index=False)


# =========================
# Ticker-Liste laden
# =========================
def load_ticker_list() -> dict:
    """
    Lädt die bereits von 01_assets.ipynb gefilterten und
    gespeicherten Assets.

    Rückgabe: {date: DataFrame mit asset_id, symbol, market_symbol, ...}
    """
    tickersperday = {}
    # ── assets.parquet laden → enthält symbol ──
    df_assets = pd.read_parquet(ASSETS_PATH)
    print(f"[TICKER] Assets geladen: {len(df_assets)} Einträge")

    # ── asset_daily laden → alle Dateien zusammenführen ──
    parquet_files = sorted(ASSET_DAILY_PATH.glob("*.parquet"))
    print(f"[TICKER] Gefundene Daily-Dateien: {[f.name for f in parquet_files]}")

    df_daily = pd.concat(
        [pd.read_parquet(f) for f in parquet_files],
        ignore_index=True
    )
    df_daily["date"] = pd.to_datetime(df_daily["date"]).dt.date

    for date in sorted(df_daily["date"].unique()):
        count = len(df_daily[df_daily["date"] == date])
        print(f"  {date}: {count} Zeilen")
        df_date = df_daily[df_daily["date"] == date].copy()

        # ── Join: asset_id → symbol ──
        df = df_date.merge(
            df_assets[["asset_id", "symbol", "name"]],
            on="asset_id",
            how="left"
        )

        # ── market_symbol bauen: SYMBOL + USDT ──
        # z.B. "BTC" → "BTCUSDT"
        # Nur Assets mit Binance-Volumen > $50M → sind auf Binance handelbar
        df = df[df["binance_volume_24h_usd"].notna()]
        df = df[df["binance_volume_24h_usd"] > 50_000_000]
        df["market_symbol"] = df["symbol"].str.upper() + "USDT"

        tickersOfOneDay = df[[
            "asset_id", "symbol", "name", "market_symbol",
            "market_cap_usd", "global_volume_24h_usd", "binance_volume_24h_usd"
        ]].reset_index(drop=True)
        tickersperday[date] = tickersOfOneDay

        print(f"\n[TICKER] Ticker-Liste {date} ({len(tickersOfOneDay)} Assets):")
        display_df = tickersOfOneDay[["symbol", "market_symbol", "market_cap_usd", "binance_volume_24h_usd"]].copy()
        display_df["market_cap_usd"]         = display_df["market_cap_usd"].apply(lambda x: f"${x:>15,.0f}")
        display_df["binance_volume_24h_usd"] = display_df["binance_volume_24h_usd"].apply(lambda x: f"${x:>13,.0f}")
        print(display_df.to_string(index=True))

    print("\n[TICKER] Übersicht aller Tage:")
    for date, df in sorted(tickersperday.items()):
        print(f"\n  {date} ({len(df)} Ticker):")
        print("    " + ",  ".join(df["market_symbol"].tolist()))

    return tickersperday


# =========================
# Hauptfunktion
# =========================
def fetch_ohlcv_for_month() -> None:
    """
    Lädt historische 5m OHLCV-Daten von Binance für ALLE Tage
    des jeweiligen Monats — nicht nur für den Snapshot-Tag.

    Pro Monat wird der asset_daily-Snapshot als Ticker-Quelle verwendet.
    Bereits vorhandene Tage werden übersprungen (kein Re-Fetch).
    """
    print("=" * 60)
    print("Starte OHLCV-Datenbeschaffung")
    print("=" * 60)

    # Schritt 1: Ticker-Snapshots laden {snapshot_date: DataFrame}
    tickersperday = load_ticker_list()

    for snapshot_date, df_tickers in sorted(tickersperday.items()):
        symbols = df_tickers["market_symbol"].tolist()
        year    = snapshot_date.year
        month   = snapshot_date.month

        # ── Alle Tage des Monats generieren ──
        month_start   = pd.Timestamp(year=year, month=month, day=1)
        days_in_month = month_start.days_in_month
        all_dates     = pd.date_range(month_start, periods=days_in_month, freq="D").date

        # ── Bereits vorhandene Tage aus Parquet lesen ──
        out = ASSET_OHLCV_5M_DIR / f"ohlcv_{year:04d}-{month:02d}.parquet"
        existing_dates = set()
        if out.exists():
            df_ex = pd.read_parquet(out, columns=["timestamp"])
            df_ex["date"] = pd.to_datetime(df_ex["timestamp"]).dt.date
            existing_dates = set(df_ex["date"].unique())

        print(f"\n[FETCH] {year}-{month:02d}: {days_in_month} Tage, {len(symbols)} Ticker")
        print(f"  Bereits vorhanden: {sorted(existing_dates)}")

        for current_date in all_dates:
            if current_date in existing_dates:
                print(f"  [SKIP] {current_date}: bereits vorhanden")
                continue

            print(f"\n  [FETCH] {current_date}: {len(symbols)} Ticker")
            all_rows = []

            for market_symbol in symbols:
                try:
                    raw = public_get("/api/v3/klines", params={
                        "symbol":    market_symbol,
                        "interval":  INTERVAL,
                        "startTime": int(pd.Timestamp(current_date, tz="UTC").timestamp() * 1000),
                        "endTime":   int((pd.Timestamp(current_date, tz="UTC") + pd.Timedelta(days=1)).timestamp() * 1000),
                        "limit":     1000,  # 5m = max 288 Kerzen pro Tag
                    })

                    for row in raw:
                        all_rows.append({
                            "date"  :        pd.to_datetime(int(row[0]), unit="ms", utc=True).date(),  
                            "timestamp":     pd.to_datetime(int(row[0]), unit="ms", utc=True),
                            "market_symbol": market_symbol,
                            "open":          float(row[1]),
                            "high":          float(row[2]),
                            "low":           float(row[3]),
                            "close":         float(row[4]),
                            "volume_base":   float(row[5]),
                            "volume_quote":  float(row[7]),
                            "trade_count":   int(row[8]),
                        })

                    print(f"    [OK] {market_symbol}: {len(raw)} Kerzen")
                    time.sleep(SLEEP_BETWEEN_REQUESTS)

                except Exception as e:
                    print(f"    [FEHLER] {market_symbol}: {e}")

            if all_rows:
                print(f"  [SAVE] {len(all_rows)} Zeilen → Parquet")
                json_to_parquet(all_rows)
            else:
                print(f"  [SKIP] Keine Daten für {current_date}")

    print("\n" + "=" * 60)
    print("OHLCV-Datenbeschaffung abgeschlossen")
    print("=" * 60)


# =========================
# Ausführen
# =========================
fetch_ohlcv_for_month()

Starte OHLCV-Datenbeschaffung
[TICKER] Assets geladen: 100 Einträge
[TICKER] Gefundene Daily-Dateien: ['asset_daily.parquet']
  2026-01-01: 100 Zeilen

[TICKER] Ticker-Liste 2026-01-01 (10 Assets):
  symbol market_symbol      market_cap_usd binance_volume_24h_usd
0    BTC       BTCUSDT  $1,747,837,206,254         $  552,991,635
1    ETH       ETHUSDT    $358,047,402,286         $  406,547,667
2    XRP       XRPUSDT    $111,584,107,851         $   76,213,965
3    BNB       BNBUSDT    $118,876,603,740         $   69,705,177
4   USDC      USDCUSDT    $ 75,371,182,421         $  716,113,452
5    SOL       SOLUSDT    $ 70,088,211,901         $  217,068,948
6   DOGE      DOGEUSDT    $ 19,713,328,746         $  109,222,808
7   AVAX      AVAXUSDT    $  5,293,277,789         $   57,578,512
8    ZEC       ZECUSDT    $  8,434,509,949         $   78,157,065
9   PEPE      PEPEUSDT    $  1,696,739,141         $   62,255,109
  2026-01-02: 100 Zeilen

[TICKER] Ticker-Liste 2026-01-02 (14 Assets):
   s

In [28]:
# =========================
# OHLCV Schnellübersicht
# =========================

print("=" * 70)
print("OHLCV DATEI-ÜBERSICHT")
print("=" * 70)

for file in sorted(ASSET_OHLCV_5M_DIR.glob("ohlcv_*.parquet")):
    df = pd.read_parquet(file)
    df["date"] = pd.to_datetime(df["timestamp"]).dt.date

    day_ticker  = df.groupby("date")["market_symbol"].nunique()
    day_candles = df.groupby("date").size()

    print(f"\n📄 {file.name}")
    print(f"   Tage gesamt : {df['date'].nunique()}")
    print(f"   Ticker Ø    : {day_ticker.mean():.0f}  (min {day_ticker.min()}, max {day_ticker.max()})")
    print()
    print(f"   {'Datum':<14} {'Ticker':>8} {'Kerzen':>10} {'Kerzen/Ticker':>15}")
    print(f"   {'-'*14} {'-'*8} {'-'*10} {'-'*15}")

    for date in sorted(day_ticker.index):
        ticker  = day_ticker[date]
        candles = day_candles[date]
        ratio   = candles / ticker if ticker > 0 else 0
        status  = "✅" if ratio >= 274 else "⚠️ " if ratio >= 200 else "❌"
        print(f"   {str(date):<14} {ticker:>8} {candles:>10,} {ratio:>14.0f}  {status}")

print("\n" + "=" * 70)

OHLCV DATEI-ÜBERSICHT

📄 ohlcv_2026-01.parquet
   Tage gesamt : 31
   Ticker Ø    : 12  (min 10, max 57)

   Datum            Ticker     Kerzen   Kerzen/Ticker
   -------------- -------- ---------- ---------------
   2026-01-01           10      2,880            288  ✅
   2026-01-02           10      2,880            288  ✅
   2026-01-03           10      2,880            288  ✅
   2026-01-04           10      2,880            288  ✅
   2026-01-05           10      2,880            288  ✅
   2026-01-06           10      2,880            288  ✅
   2026-01-07           10      2,880            288  ✅
   2026-01-08           10      2,880            288  ✅
   2026-01-09           10      2,880            288  ✅
   2026-01-10           10      2,880            288  ✅
   2026-01-11           10      2,880            288  ✅
   2026-01-12           10      2,880            288  ✅
   2026-01-13           10      2,880            288  ✅
   2026-01-14           10      2,880            288  ✅
  